In [ ]:
import numpy as np, pandas as pd, os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Experiment A3 — Multi-Hop Relay at 200+ Concepts

Exp 8 multi-hop used only 20 concepts and hit a ceiling (everything ~100% accurate). With 200+ concepts, genuine information pressure reveals whether dense or text protocols degrade faster across relay hops. The hypothesis: dense degrades smoothly (noise accumulates in float space), text degrades step-wise (each hop re-quantizes). Or vice versa.

In [ ]:
!pip install sentence-transformers wordfreq torch matplotlib pandas -q

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
torch.manual_seed(42)

# ── The ceiling problem being fixed ───────────────────────────────────────────
# Exp 8 multi-hop used 20 concepts → near-perfect accuracy at every hop.
# The task is too easy: any signal is enough to identify 1-of-20.
# With 200+ concepts: genuine information pressure, degradation becomes visible.
# The key hypothesis: text relay degrades STEP-WISE (each hop re-quantizes into
# a new discrete token) while dense degrades SMOOTHLY (noise accumulates
# continuously). Or vice versa — this is what we find out.

N_HOPS      = 8         # A→B→C→…→H
NOISE_SIGMAS = [0.0, 0.05, 0.1, 0.2, 0.3]
K_VALUES    = [4, 8, 16]     # dense bottleneck dimensions
N_TRAIN_AE  = 500       # concepts for training the bottleneck AE
N_EVAL      = 200       # held-out concepts for the relay test
HIDDEN_AE   = 256
N_EPOCHS_AE = 2000
print('Setup complete')

In [ ]:
!pip install sentence-transformers wordfreq -q
from sentence_transformers import SentenceTransformer
from wordfreq import top_n_list

model = SentenceTransformer('LaBSE', device='cpu')

# ── Build 700-concept vocabulary ───────────────────────────────────────────────
print('Building 700-concept vocabulary...')
words_raw = top_n_list('en', 8000)
words = [w for w in words_raw if len(w) >= 3 and w.isalpha()][:700]
concept_embs = model.encode(words, normalize_embeddings=True, batch_size=256, show_progress_bar=True)
print(f'Embedded {len(words)} concepts: {concept_embs.shape}')

# Split: train AE on first N_TRAIN_AE, eval relay on last N_EVAL
train_embs = concept_embs[:N_TRAIN_AE]
eval_embs  = concept_embs[N_TRAIN_AE:N_TRAIN_AE+N_EVAL]
eval_words = words[N_TRAIN_AE:N_TRAIN_AE+N_EVAL]
print(f'Train AE: {train_embs.shape}  |  Eval relay: {eval_embs.shape}')

In [ ]:
class BottleneckAE(nn.Module):
    def __init__(self, k, hidden=HIDDEN_AE):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(768,hidden),nn.GELU(),nn.Linear(hidden,hidden//2),nn.GELU(),nn.Linear(hidden//2,k))
        self.dec = nn.Sequential(nn.Linear(k,hidden//2),nn.GELU(),nn.Linear(hidden//2,hidden),nn.GELU(),nn.Linear(hidden,768))
    def encode(self, x): return self.enc(x)
    def forward(self, x):
        z = self.enc(x)
        return F.normalize(self.dec(z), dim=-1), z

def train_ae(k, n_epochs=N_EPOCHS_AE):
    X = torch.tensor(train_embs, dtype=torch.float32)
    m = BottleneckAE(k)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for ep in range(n_epochs):
        m.train()
        recon, z = m(X)
        recon_loss = (1.0 - (recon*X).sum(-1)).mean()
        ref_loss   = F.cross_entropy(recon @ X.T, torch.arange(len(X)))
        (recon_loss + ref_loss).backward(); opt.step(); opt.zero_grad()
    return m

print('Training bottleneck AEs...')
ae_models = {}
for k in K_VALUES:
    print(f'  k={k}...')
    ae_models[k] = train_ae(k)
    ae_models[k].eval()
print('All AEs trained')

# ── Multi-hop relay functions ─────────────────────────────────────────────────
def dense_relay(embs, ae_model, n_hops, noise_sigma):
    """Chain of relays: each hop compresses → adds noise → decompresses."""
    X = torch.tensor(embs, dtype=torch.float32)
    with torch.no_grad():
        current = X.clone()
        for hop in range(n_hops):
            # Compress
            z = ae_model.encode(current)
            # Add noise to compressed representation
            if noise_sigma > 0:
                z = z + torch.randn_like(z) * noise_sigma
            # Decompress
            current = F.normalize(ae_model.dec(z), dim=-1)
    return current.numpy()

def text_relay(embs, n_hops, noise_sigma):
    """Text relay: each hop re-embeds (simulated by adding noise to full embedding).
    Text is modelled as the LaBSE embedding with token-level perturbation.
    We simulate word-level noise as additive Gaussian on the 768-d embedding,
    but at a HIGHER rate per-hop than dense (text carries redundancy that smooths
    small noise, but character errors degrade sharply).
    We use the same Gaussian model but on the full 768-d space."""
    current = embs.copy()
    for hop in range(n_hops):
        if noise_sigma > 0:
            # Text noise: simulate discrete re-quantization as higher-variance perturbation
            # Each hop: noise is applied to the full embedding (simulating re-encoding)
            current = current + np.random.randn(*current.shape) * noise_sigma
            norms = np.linalg.norm(current, axis=1, keepdims=True).clip(1e-9)
            current = current / norms
    return current

def identification_accuracy(relayed_embs, reference_embs, n_sample=200):
    """How many concepts can be correctly identified after n_hops relays?"""
    rng = np.random.default_rng(7)
    idx = rng.choice(len(reference_embs), min(n_sample, len(reference_embs)), replace=False)
    sims = relayed_embs[idx] @ reference_embs.T
    correct = (sims.argmax(axis=1) == idx).mean()
    return float(correct)

In [ ]:
print('Running multi-hop relay sweeps...')
results = []

# Baseline: identity (0 hops, 0 noise)
base_acc = identification_accuracy(eval_embs, eval_embs)
print(f'Baseline (no relay): {base_acc:.3f}')

for sigma in NOISE_SIGMAS:
    for n_hops in range(0, N_HOPS+1):
        # Dense protocols
        for k in K_VALUES:
            relayed = dense_relay(eval_embs, ae_models[k], n_hops, sigma)
            acc = identification_accuracy(relayed, eval_embs)
            results.append({'protocol':f'Dense k={k}','k':k,'n_hops':n_hops,'sigma':sigma,'accuracy':acc})
        # Text protocol (Gaussian on full embedding)
        relayed_text = text_relay(eval_embs, n_hops, sigma)
        acc_text = identification_accuracy(relayed_text, eval_embs)
        results.append({'protocol':'Text (full emb)','k':768,'n_hops':n_hops,'sigma':sigma,'accuracy':acc_text})
        # Raw LaBSE (no compression, just noise)
        relayed_raw = text_relay(eval_embs, n_hops, sigma * 0.5)  # LaBSE more resilient
        acc_raw = identification_accuracy(relayed_raw, eval_embs)
        results.append({'protocol':'Raw LaBSE','k':768,'n_hops':n_hops,'sigma':sigma,'accuracy':acc_raw})

import pandas as pd
df = pd.DataFrame(results)
print(f'Sweep complete: {len(df)} data points')
print(df[df.sigma==0.1].groupby(['protocol','n_hops'])['accuracy'].mean().unstack().round(3).to_string())

In [ ]:
import pandas as pd
df = pd.read_csv('/dev/stdin') if False else df  # use in-memory df

fig, axes = plt.subplots(1, len(NOISE_SIGMAS), figsize=(5*len(NOISE_SIGMAS), 5), sharey=True)
COLORS = {f'Dense k={k}': c for k,c in zip(K_VALUES,['#1D9E75','#378ADD','#EF9F27'])}
COLORS.update({'Text (full emb)': '#E24B4A', 'Raw LaBSE': '#7F77DD'})

for ax, sigma in zip(axes, NOISE_SIGMAS):
    sub = df[df.sigma==sigma]
    for protocol, color in COLORS.items():
        psub = sub[sub.protocol==protocol].sort_values('n_hops')
        style = '--' if 'Text' in protocol else ('-' if 'Dense' in protocol else ':')
        ax.plot(psub['n_hops'], psub['accuracy'], style, color=color, label=protocol,
                linewidth=2, marker='o', markersize=5)
    ax.set_title(f'sigma={sigma}')
    ax.set_xlabel('Number of relay hops')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.05, 1.05)

axes[0].set_ylabel('Identification accuracy')
# Single legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(COLORS), fontsize=9,
           bbox_to_anchor=(0.5, -0.05))

plt.suptitle(f'Experiment A3 — Multi-Hop Relay at {N_EVAL} Concepts\n'
             'Does dense protocol degrade smoothly while text degrades step-wise?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp_a3_multihop.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary ──────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('EXPERIMENT A3 — MULTI-HOP RELAY SUMMARY')
print('='*60)
print(f'Concepts: {N_EVAL} held-out | Hops: 0-{N_HOPS} | Noise: {NOISE_SIGMAS}')
print()
print('Accuracy at hop=8, sigma=0.1:')
subset = df[(df.n_hops==N_HOPS)&(df.sigma==0.1)][['protocol','accuracy']].drop_duplicates('protocol')
print(subset.to_string(index=False))
print()
best_dense = df[(df.n_hops==N_HOPS)&(df.sigma==0.1)&(df.protocol.str.startswith('Dense'))]['accuracy'].max()
text_val   = df[(df.n_hops==N_HOPS)&(df.sigma==0.1)&(df.protocol=='Text (full emb)')]['accuracy'].values[0]
print(f'Dense (best k) advantage over text at {N_HOPS} hops: {best_dense-text_val:+.3f}')
if best_dense > text_val + 0.05:
    print('VERDICT: Dense degrades more gracefully than text over many hops.')
elif text_val > best_dense + 0.05:
    print('VERDICT: Text degrades more gracefully — dense compression loses information each hop.')
else:
    print('VERDICT: Similar degradation rate — compression bottleneck and text noise balance out.')